# import important libraries 

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px 
import warnings 
warnings.filterwarnings("ignore")
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import accuracy_score 
from sklearn.metrics import accuracy_score,recall_score,f1_score ,classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import re 

# Load Data 

In [ ]:
data=pd.read_csv('Restaurant_Reviews.tsv',sep='\t')

# Data Overview 

In [ ]:
data 

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.describe(include=['object'])

there 996 unique means that there are duplicated in column review 

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data.isna().sum()

# Explore data 

In [ ]:
data.columns

In [ ]:
count_like=data.Liked.value_counts()
df_like=pd.DataFrame(count_like).reset_index()
df_like


column target(liked) is highly balnced data

In [ ]:
fig = px.pie(df_like, names='Liked', values='count',color_discrete_sequence=['lightblue','lightpink'],title='Distribution of Likes')
fig.update_traces(textinfo='label+percent')
fig.show()

In [ ]:
data['Reviews letter counts']=data.Review.apply(len)

In [ ]:
data.Review.apply(len).max()

In [ ]:
data 

In [ ]:
data.iloc[data.Review.apply(len).idxmax()]

In [ ]:
# data.iloc[data.Review.apply(len).argmax()][0] will be the same result 
data.iloc[data.Review.apply(len).idxmax()][0]

# Pre-Processing Data

In [ ]:
data.columns=data.columns.str.lower()

In [ ]:
data

In [ ]:
s=data.review[0]
s

In [ ]:
# #bec it can be have meaning as ! or ? 
# import string
# string.punctuation

In [ ]:
nltk.download('stopwords')

In [ ]:
print(stopwords.words('english'))

to remove stopwords you must make all words lower and split()

In [ ]:
# substiute all not letter with space 
statments=re.sub('[^a-zA-Z]',' ',s)
statments

In [ ]:
statments=statments.lower()
statments

In [ ]:
# if not splited it wil;l lioop on the letter
splited=statments.split()
splited

In [ ]:
temp=[]
for word in splited:
    if word not in stopwords.words('english'):
        temp.append(word)
        
temp        

In [ ]:
# will join list elements (words) with space to return it again sentence (str)
s=' '.join(temp)
s

In [ ]:
ps=PorterStemmer()

the PorterStemmer is a valuable tool in NLP for preprocessing text data. By simplifying words to their root forms, it helps improve the efficiency and effectiveness of various NLP tasks, making it easier to analyze and interpret text data

In [ ]:
s= ps.stem(s)

In [ ]:
print(s)

In [ ]:
ss=SnowballStemmer('english')

In [ ]:
ss= ss.stem(s)
print(ss)

In [ ]:
ls=LancasterStemmer()
ls= ls.stem(s)
print(ls)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

sklearn.feature_extraction.text module is a crucial tool in NLP for converting a collection of text documents into a matrix of token counts. 

In [ ]:
vectorizer = CountVectorizer()


In [ ]:
vectorizer.fit_transform(s.split()).toarray()

In [ ]:
corpus=[]
for i in data['review']:
    s=re.sub('[^a-zA-Z]',' ',i)
    s=s.lower()
    s=s.split()
    s=[word for word in s if word not in stopwords.words('english') ]
    s=' '.join(s)
    s=ps.stem(s)
    corpus.append(s)
print(corpus)


In [ ]:
vec=vectorizer.fit_transform(corpus).toarray()
vec.shape

# split features and target 

In [ ]:
x=vec
y=data.liked

In [ ]:
x

In [ ]:
y

# split data train & test 

In [ ]:
x_train , x_test ,y_train ,y_test =train_test_split(x,y,test_size=0.33,random_state=43)

# Bulid model 

In [ ]:
columns=['LogisticRegression','DecisionTreeClassifier','RandomForestClassifier','SVC',"MNB",'XGB']
log_reg=LogisticRegression()
dtc=DecisionTreeClassifier()
rfc=RandomForestClassifier()
svc=SVC()
MNB=MultinomialNB()
xgb=xgb.XGBClassifier()

In [ ]:
test_score=[]
train_score=[]
rec_score=[]
f1_score=[]

In [ ]:
def all(model):
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    accuracy_test=accuracy_score(y_pred,y_test)*100
    accuracy_train=model.score(x_train,y_train)*100
    recall_result=recall_score(y_pred,y_test)*100
    test_score.append(accuracy_test)
    train_score.append(accuracy_train)
    rec_score.append(recall_result)

    print('Accuracy after train the model is :',accuracy_train)
    print('Accuracy after test the model is :',accuracy_test)
    print('Result recall score is :',recall_result)


In [ ]:
all(log_reg)

In [ ]:
all(dtc)

In [ ]:
all(rfc)

In [ ]:
all(svc)

In [ ]:
all(MNB)

In [ ]:
all(xgb)

In [ ]:
print(test_score)
print(train_score)
print(rec_score)

In [ ]:
plt.figure(figsize=(15, 8))
bar_width = 0.2
xpos=np.arange(len(columns))
bars1=plt.bar(xpos - 0.3, train_score, width=bar_width, label="train_score",color='lightblue')
bars2=plt.bar(xpos - 0.1, test_score, width=bar_width, label="test_score",color='lightpink')
bars3=bars=plt.bar(xpos + 0.1, rec_score, width=bar_width, label="recall_score",color='lightgreen')

plt.xticks(xpos, columns)
plt.legend()
plt.xlabel("Models",fontsize=15)
plt.ylabel("Scores",fontsize=15)
plt.title("Model Performance Comparison",fontsize=30)
plt.show()
print("logistic Rgression Model Results:","train_score:", train_score[0],"test_score:", test_score[0],"rec_score:", rec_score[0])
print("Decision Tree Model Results:","train_score:", train_score[1],"test_score:", test_score[1],"rec_score:", rec_score[1]) 
print("Random Forest Regressor:","train_score:", train_score[2],"test_score:", test_score[2],"rec_score:", rec_score[2])
print("SVC Model Results:","train_score:", train_score[3],"test_score:", test_score[3],"rec_score:", rec_score[3]) 
print("Multinomial Model Results:","train_score:", train_score[4],"test_score:", test_score[4],"rec_score:", rec_score[4]) 
print("XGB Model Results:","train_score:", train_score[5],"test_score:", test_score[5],"rec_score:", rec_score[5]) 



# 📊 **Model Performance Comparison**

## **Overfitting:**
- **Decision Tree** and **Random Forest** show high training scores (> 99%) but significantly lower test scores, indicating potential overfitting.
- **Logistic Regression**, **SVC**, and **XGBoost** exhibit more balanced scores, suggesting less overfitting.

## 📈 **Accuracy (Test Scores):**
- **SVC** achieves the highest test score of **76.60%**, making it the most accurate model.
- **Logistic Regression** follows closely with a test score of **75.38%**.

## 📊 **Recall:**
- **SVC** also leads in recall with a score of **85.09%**, indicating it is the best at identifying positive instances.
- **Random Forest** ranks second with a recall score of **80.83%**.

## 🏆 **Best Model:**
- **SVC** emerges as the best model with the highest test accuracy and recall, demonstrating its ability to generalize well to unseen data while effectively identifying positive cases.

## **XGBoost:** 
- Although **XGBoost** has the lowest overfitting, its accuracy is slightly lower than **SVC** and **Logistic Regression**, making it a strong contender for unseen data despite the lower score.

---

*📝 A detailed analysis of model performance on training and test datasets to help choose the best-performing model for our problem.*


In [ ]:
import pickle

# Save the vectorizer (CountVectorizer fitted on your corpus)
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save your best model — using RandomForestClassifier (change if you prefer another)
with open("model.pkl", "wb") as f:
    pickle.dump(rfc, f)

print("✅ vectorizer.pkl and model.pkl saved successfully!")